# Метод 11/66: Doroshenko

Восстановление ППА Cs-137 | Doroshenko | Iterative

Координатно-обновляемый метод. Numba JIT.

In [1]:
import sys, os, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.expanduser('~/my-project/soilactivity/src'))
sys.path.insert(0, os.path.expanduser('~/my-project/bssunfold/src'))
from soilactivity.fredholm import build_fredholm_matrix_no_vis, vector_to_raster
from soilactivity.radionuclides import KERMA_CONSTANTS
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## Постановка задачи

Сетка 12×12, ячейка 10 м, два гауссовых пятна Cs-137, шум 5%.

In [2]:
np.random.seed(42)
NX, NY = 12, 12
CELL = 10.0; HEIGHT = 1.0; NOISE = 0.05
KERMA = KERMA_CONSTANTS['Cs-137']; W = 1.0
cx, cy = np.meshgrid(np.arange(NX)*CELL+CELL/2, np.arange(NY)*CELL+CELL/2)
A_true = (500*np.exp(-((cx-40)**2+(cy-50)**2)/(2*15**2))
        + 300*np.exp(-((cx-80)**2+(cy-30)**2)/(2*20**2)) + 10.0)
a_true = A_true.ravel('C')
N = NX * NY
F = build_fredholm_matrix_no_vis(NX, NY, CELL, HEIGHT, KERMA, W)
P_true = F @ a_true
rng = np.random.default_rng(42)
P_meas = np.maximum(P_true * (1 + NOISE*rng.standard_normal(N)), 1e-30)
A_mat, b_vec = F.copy(), P_meas.copy()
x0 = np.ones(N) * np.mean(P_meas) / np.mean(np.diag(A_mat))
print(f'N={N}, cond(F)={np.linalg.cond(F):.2e}')

N=144, cond(F)=1.15e+00


In [3]:
from bssunfold.core import solve_doroshenko

## Решение методом Doroshenko

In [4]:
t0 = time.time()
x_recon = np.zeros(N)
success = False
err_msg = ''
n_iter = None

try:
    r = solve_doroshenko(A_mat, b_vec, x0, **{"max_iterations": 200, "tolerance": 1e-8, "regularization": 0.01})
    if isinstance(r, tuple):
        x_recon = np.array(r[0], dtype=float)
        n_iter = r[1] if len(r) > 1 else None
    else:
        x_recon = np.array(r, dtype=float)
    success = True
except Exception as e:
    err_msg = str(e)

elapsed = time.time() - t0
status = 'OK' if success else 'FAIL'
print(f'Status: {status} | Time: {elapsed:.2f}s')
if n_iter is not None:
    print(f'Iterations: {n_iter}')
if err_msg:
    print(f'Error: {err_msg[:100]}')

Status: OK | Time: 3.22s
Iterations: 12


## Результаты

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
vmax = max(A_true.max(), vector_to_raster(np.maximum(x_recon,0), NY, NX).max()) * 1.05
im0 = axes[0].imshow(A_true, origin='lower', cmap='hot', extent=[0,NX*CELL,0,NY*CELL], vmin=0, vmax=vmax)
axes[0].set_title('Истинная активность', fontsize=12)
axes[0].set_xlabel('X, м'); axes[0].set_ylabel('Y, м')
plt.colorbar(im0, ax=axes[0], label='Бк/м2', shrink=0.8)
im1 = axes[1].imshow(vector_to_raster(np.maximum(x_recon,0), NY, NX), origin='lower', cmap='hot',
                      extent=[0,NX*CELL,0,NY*CELL], vmin=0, vmax=vmax)
axes[1].set_title('Восстановление (Doroshenko)', fontsize=10)
axes[1].set_xlabel('X, м'); axes[1].set_ylabel('Y, м')
plt.colorbar(im1, ax=axes[1], label='Бк/м2', shrink=0.8)
err_map = np.abs(vector_to_raster(np.maximum(x_recon,0), NY, NX) - A_true)
im2 = axes[2].imshow(err_map, origin='lower', cmap='viridis', extent=[0,NX*CELL,0,NY*CELL])
axes[2].set_title('Абсолютная ошибка', fontsize=12)
axes[2].set_xlabel('X, м'); axes[2].set_ylabel('Y, м')
plt.colorbar(im2, ax=axes[2], label='Бк/м2', shrink=0.8)
plt.tight_layout()
plt.savefig('11_doroshenko.png', dpi=120, bbox_inches='tight')
plt.show()

In [6]:
x_nn = np.maximum(x_recon, 0)
rmse = np.sqrt(np.mean((x_nn - a_true)**2))
rel = np.mean(np.abs(x_nn - a_true) / (np.abs(a_true)+1e-30)) * 100
res = np.linalg.norm(A_mat @ x_nn - b_vec)
print('=' * 50)
print('  Метод: Doroshenko (#11')
print('  Категория: '+cat)
print('=' * 50)
print(f'  RMSE:               {rmse:.2f} Бк/м2')
print(f'  Средняя отн. ошибка: {rel:.1f} %%')
print(f'  ||Ax-b||:           {res:.3e}')
print(f'  Время:              {elapsed:.2f} с')
print('  Успех:              Да')
if err_msg:
    print(f'  Ошибка:             {err_msg[:80]}')
print('=' * 50)

  Метод: Doroshenko (#11


NameError: name 'cat' is not defined

## Сравнение с Фурье-сверткой

In [ ]:
from numpy.fft import fft2, ifft2

F_diag = np.diag(F).reshape(NY, NX)
P_2d = b_vec.reshape(NY, NX)
F_fft = fft2(F_diag)
P_fft = fft2(P_2d)
alpha_fft = 1e-6 * np.max(np.abs(F_fft)**2)
a_fft = np.real(ifft2(P_fft * np.conj(F_fft) / (np.abs(F_fft)**2 + alpha_fft)))
a_fft = np.maximum(a_fft, 0)
rmse_fft = np.sqrt(np.mean((a_fft.ravel() - a_true)**2))
print(f'Fourier deconv RMSE: {rmse_fft:.2f} Bq/m2')
rmse_m = np.sqrt(np.mean((np.maximum(x_recon,0) - a_true)**2))
print('bssunfold Doroshenko RMSE: {rmse_m:.2f} Bq/m2')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
vmax = max(A_true.max(), a_fft.max()) * 1.05
im0 = axes[0].imshow(A_true, origin='lower', cmap='hot', extent=[0,NX*CELL,0,NY*CELL], vmin=0, vmax=vmax)
axes[0].set_title('Истинная активность'); axes[0].set_xlabel('X, м'); axes[0].set_ylabel('Y, м')
plt.colorbar(im0, ax=axes[0], label='Бк/м2', shrink=0.8)
im1 = axes[1].imshow(a_fft, origin='lower', cmap='hot', extent=[0,NX*CELL,0,NY*CELL], vmin=0, vmax=vmax)
axes[1].set_title('Фурье-деконволюция'); axes[1].set_xlabel('X, м'); axes[1].set_ylabel('Y, м')
plt.colorbar(im1, ax=axes[1], label='Бк/м2', shrink=0.8)
im2 = axes[2].imshow(vector_to_raster(np.maximum(x_recon,0), NY, NX), origin='lower', cmap='hot',
                      extent=[0,NX*CELL,0,NY*CELL], vmin=0, vmax=vmax)
axes[2].set_title('bssunfold (Doroshenko)'); axes[2].set_xlabel('X, м'); axes[2].set_ylabel('Y, м')
plt.colorbar(im2, ax=axes[2], label='Бк/м2', shrink=0.8)
plt.tight_layout()
plt.savefig('11_doroshenko_fft.png', dpi=120, bbox_inches='tight')
plt.show()